# SVV-kamera – YOLO26 trening på Google Colab

**Steg:**
1. Koble til Google Drive
2. Last opp `dataset.zip` til Drive
3. Kjør cellene i rekkefølge
4. Beste vekter lagres til `MyDrive/svv_yolo/best.pt`

**Krav:** Kjør med GPU-akselerasjon: `Rediger → Notatbokinnstillinger → T4 GPU`

In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ Ingen GPU funnet – bytt runtime til T4 GPU')

Wed Feb 25 13:43:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:

%pip install ultralytics -q
print('ultralytics installert')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 35.3 MB/s eta 0:00:00
✅ ultralytics installert


In [3]:
# ── 3. Koble til Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive montert')

Mounted at /content/drive
✅ Drive montert


In [4]:
# ── 4. Pakk ut dataset ────────────────────────────────────────────────────────
# Last opp dataset.zip til Google Drive først:
#   Gå til drive.google.com → Min Disk → legg inn dataset.zip
#
# Lag dataset.zip lokalt (kjør i terminalen din):
#   cd /Users/sondre/svv_kamera/Master-s-Thesis-V26---Sondre-Eirik/Model
#   zip -r dataset.zip dataset/ dataset.yaml

import zipfile, os
from pathlib import Path

ZIP_PATH    = '/content/drive/MyDrive/dataset.zip'   # ← juster hvis du la den i en mappe
EXTRACT_DIR = '/content/svv_dataset'

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)

print(f'✅ Dataset pakket ut til {EXTRACT_DIR}')
!find {EXTRACT_DIR} -maxdepth 3 -type d

✅ Dataset pakket ut til /content/svv_dataset
/content/svv_dataset
/content/svv_dataset/dataset
/content/svv_dataset/dataset/images
/content/svv_dataset/dataset/images/train
/content/svv_dataset/dataset/images/val
/content/svv_dataset/dataset/labels
/content/svv_dataset/dataset/labels/train
/content/svv_dataset/dataset/labels/val


In [5]:
# ── 5. Fiks dataset.yaml med korrekte Colab-stier ────────────────────────────
import yaml

YAML_SRC = f'{EXTRACT_DIR}/dataset.yaml'

with open(YAML_SRC) as f:
    cfg = yaml.safe_load(f)

cfg['path']  = f'{EXTRACT_DIR}/dataset'
cfg['train'] = 'images/train'
cfg['val']   = 'images/val'

YAML_OUT = '/content/dataset_colab.yaml'
with open(YAML_OUT, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True)

print('✅ dataset_colab.yaml:')
!cat {YAML_OUT}

✅ dataset_colab.yaml:
names:
  0: Buss
  1: Long combination vehicle
  2: Semi-trailer
  3: bicycle
  4: car
  5: motorcycle
  6: person
  7: truck
nc: 8
path: /content/svv_dataset/dataset
train: images/train
val: images/val


In [6]:
# ── 6. Trening ────────────────────────────────────────────────────────────────
from ultralytics import YOLO

MODEL   = 'yolo26n.pt'   # nano – rask og lett
EPOCHS  = 100
IMGSZ   = 640            # full oppløsning – GPU har nok minne
BATCH   = 16             # T4 har 16 GB – trygt med batch 16

model = YOLO(MODEL)

results = model.train(
    data      = YAML_OUT,
    epochs    = EPOCHS,
    imgsz     = IMGSZ,
    batch     = BATCH,
    device    = 0,          # GPU
    patience  = 20,
    pretrained= True,
    project   = '/content/runs',
    name      = 'svv_kamera',
    exist_ok  = True,
    save      = True,
    plots     = True,
)

print('\n Trening ferdig!')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.16 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=Fa

In [7]:
# ── 7. Validering ─────────────────────────────────────────────────────────────
metrics = model.val()
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

Ultralytics 8.4.16 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,376,396 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1372.3±507.8 MB/s, size: 29.2 KB)
val: Scanning /content/svv_dataset/dataset/labels/val.cache... 23 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 23/23 6.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.0it/s 0.7s
                   all         23        111       0.63      0.217      0.334      0.273
                  Buss          5          5          1          0      0.552      0.452
                   car         22        100      0.521       0.87      0.752      0.623
                person          3          3          1          0          0          0
                 truck          3          3          0          0     0.0307     0.0179
Speed: 2.7ms preprocess, 19.0ms inference,

In [8]:
# ── 8. Kopier beste vekter til Google Drive ───────────────────────────────────
import shutil

BEST_PT  = '/content/runs/svv_kamera/weights/best.pt'
DRIVE_OUT = '/content/drive/MyDrive/svv_yolo/'

os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copy2(BEST_PT, DRIVE_OUT + 'best.pt')
print(f'✅ best.pt lagret til {DRIVE_OUT}best.pt')

# Kopier også treningsplottene
for f in Path('/content/runs/svv_kamera').glob('*.png'):
    shutil.copy2(f, DRIVE_OUT)
print('✅ Treningsplott kopiert til Drive')

✅ best.pt lagret til /content/drive/MyDrive/svv_yolo/best.pt
✅ Treningsplott kopiert til Drive


In [9]:
# ── 9. Last ned best.pt direkte til PC (alternativ til Drive) ─────────────────
from google.colab import files
files.download(BEST_PT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# ── 10. Test modellen på video ────────────────────────────────────────────────
# Last opp en videofil til Colab, eller bruk en fra Google Drive
# Alternativ A: Last opp fra PC
from google.colab import files as colab_files
import os

print("Last opp en videofil (mp4, avi, mov):")
uploaded = colab_files.upload()
video_path = list(uploaded.keys())[0]
print(f"✅ Video lastet opp: {video_path}")



Last opp en videofil (mp4, avi, mov):


Saving Fv587_Haukeland_1229050_03.mp4 to Fv587_Haukeland_1229050_03.mp4
✅ Video lastet opp: Fv587_Haukeland_1229050_03.mp4


In [16]:
# ── 11. Kjør inferens på video ───────────────────────────────────────────────
from ultralytics import YOLO

BEST_PT    = '/content/drive/MyDrive/svv_yolo/best.pt'
OUTPUT_DIR = '/content/drive/MyDrive/svv_yolo/video_results'

model = YOLO(BEST_PT)

results = model.predict(
    source   = video_path,
    save     = True,
    project  = OUTPUT_DIR,
    name     = 'test',
    conf     = 0.25,
    imgsz    = 640,
    device   = 0,
    exist_ok = True,
)

print(f'✅ Annotert video lagret til {OUTPUT_DIR}/test/')




WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/2982) /content/Fv587_Haukeland_1229050_03.mp4: 384x640 3 cars, 21.9ms
video 1/1 (frame 2/2982) /content/Fv587_Haukeland_1229050_03.mp4: 384x640 3 cars, 10.2ms
video 1/1 (frame 3/2982) /content/Fv587_Haukeland_1229050_03.mp4: 384x640 3 cars, 9.6ms
video 1/1 (frame 4/2982) /content/Fv587_Haukeland_1229050_03.mp4: 384x640 3 cars, 9.9ms
video 1/1 (frame 5/2982) /content/Fv587_Haukeland_1229050_03.mp4: 384x640 4 cars, 9.8ms
video 1/1 (frame

In [14]:
# ── 12. Vis noen frames fra den annoterte videoen ────────────────────────────
import cv2
from IPython.display import display, Image as IPImage
import glob, os

# Finn annotert video – søk rekursivt under OUTPUT_DIR
mp4_files = glob.glob(f'{OUTPUT_DIR}/**/*.mp4', recursive=True)
print("Filer funnet:", mp4_files)

if not mp4_files:
    print("❌ Ingen mp4 funnet. Sjekker hva som finnes i OUTPUT_DIR:")
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for f in files:
            print(os.path.join(root, f))
else:
    output_video = mp4_files[0]
    print(f"Viser frames fra: {output_video}")

    cap = cv2.VideoCapture(output_video)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Vis 6 jevnt fordelte frames
    for frame_idx in range(0, total_frames, max(1, total_frames // 6)):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            break
        _, buf = cv2.imencode('.jpg', frame)
        print(f"Frame {frame_idx}/{total_frames}")
        display(IPImage(data=buf.tobytes()))

    cap.release()
    print("✅ Ferdig")


Filer funnet: []
❌ Ingen mp4 funnet. Sjekker hva som finnes i OUTPUT_DIR:
/content/drive/MyDrive/svv_yolo/video_results/test/Kjorbekk_3000965_09.avi
